In [ ]:

import numpy as np

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score


# ==========================================
# STEP 1: GENERATE IMBALANCED DATA
# ==========================================

def make_data(seed=42):

    X, y = make_classification(
        n_samples=2000,
        n_features=20,
        n_informative=6,
        n_redundant=4,
        weights=[0.9, 0.1],
        random_state=seed
    )

    return X, y


# ==========================================
# STEP 2: SPLIT DATA
# ==========================================

def split_data(X, y, seed=0):

    return train_test_split(
        X,
        y,
        test_size=0.4,
        random_state=seed,
        stratify=y
    )


# ==========================================
# STEP 3: FIT LOGISTIC REGRESSION
# ==========================================

def fit_model(X_train, y_train, balanced=False):

    if balanced:
        model = LogisticRegression(
            max_iter=5000,
            class_weight="balanced"
        )
    else:
        model = LogisticRegression(
            max_iter=5000
        )

    model.fit(X_train, y_train)

    return model


# ==========================================
# STEP 4: F1 SCORE AT GIVEN THRESHOLD
# ==========================================

def f1_at_threshold(model, X, y, threshold=0.5):

    probabilities = model.predict_proba(X)[:, 1]

    predictions = (probabilities >= threshold).astype(int)

    score = f1_score(y, predictions)

    return float(score)


# ==========================================
# STEP 5: FIND BEST THRESHOLD
# ==========================================

def best_threshold(model, X, y, thresholds):

    best_t = None
    best_f1 = -1.0

    for threshold in thresholds:

        score = f1_at_threshold(
            model,
            X,
            y,
            threshold
        )

        if score > best_f1:
            best_f1 = score
            best_t = threshold

    return float(best_t), float(best_f1)


# ==========================================
# STEP 6: CALCULATE F1 IMPROVEMENT
# ==========================================

def f1_improvement(f1_before, f1_after):

    return float(f1_after - f1_before)


# ==========================================
# STEP 7: GENERATE DATA AND SPLIT
# ==========================================

X, y = make_data(42)

Xtr, Xte, ytr, yte = split_data(X, y, 0)

print("Dataset Shape:", X.shape)
print("Target Shape:", y.shape)
print("Minority Class Percentage:", round(y.mean() * 100, 2))


# ==========================================
# STEP 8: TRAIN NORMAL MODEL
# ==========================================

m = fit_model(
    Xtr,
    ytr,
    balanced=False
)


# ==========================================
# STEP 9: BASELINE F1 SCORE
# ==========================================

base = f1_at_threshold(
    m,
    Xte,
    yte,
    threshold=0.5
)

print("\nBaseline F1 Score:")
print(round(base, 3))


# ==========================================
# STEP 10: FIND BEST THRESHOLD
# ==========================================

ths = list(
    np.round(
        np.linspace(0.05, 0.95, 19),
        3
    )
)

bt, bf = best_threshold(
    m,
    Xte,
    yte,
    ths
)

print("\nBest Threshold:")
print(bt)

print("Best F1 Score:")
print(round(bf, 3))


# ==========================================
# STEP 11: CALCULATE IMPROVEMENT
# ==========================================

improvement = f1_improvement(
    base,
    bf
)

print("\nF1 Improvement:")
print(round(improvement, 3))


# ==========================================
# STEP 12: TRAIN BALANCED MODEL
# ==========================================

mb = fit_model(
    Xtr,
    ytr,
    balanced=True
)


# ==========================================
# STEP 13: COMPARE POSITIVE PREDICTIONS
# ==========================================

normal_positive = int(m.predict(Xte).sum())

balanced_positive = int(mb.predict(Xte).sum())

print("\nNormal Model Positive Predictions:")
print(normal_positive)

print("\nBalanced Model Positive Predictions:")
print(balanced_positive)


# ==========================================
# STEP 14: VERIFY THRESHOLD SEARCH
# ==========================================

print("\nVerification:")

print("Best threshold belongs to grid:", bt in ths)

print("Tuned F1 >= Baseline F1:", bf >= base)

print("F1 Improvement Positive:", improvement > 0)

print(
    "Balanced predicts more positives:",
    balanced_positive > normal_positive
)
